# Assignment 10: Audio classification with Speech-to-Text and Text-to-Speech
Pipeline: **audio -> CNN keyword classifier (spoken command)** and **audio -> Whisper STT (transcript)**, then **text -> gTTS (speech)**.
Metrics: accuracy / F1 / confusion matrix for the classifier, Word Error Rate (WER) for STT and for a TTS->STT round trip.

In [ ]:
!pip install -q transformers jiwer gTTS soundfile
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torchaudio
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from IPython.display import Audio, display

torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)

## 1. Dataset: Google Speech Commands (10 keywords)

In [ ]:
LABELS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
l2i = {l: i for i, l in enumerate(LABELS)}
base = torchaudio.datasets.SPEECHCOMMANDS("./", download=True)      # ~2 GB download, one time

def pick(subset_file, limit):
    # SPEECHCOMMANDS stores validation/testing lists as text files
    p = os.path.join(base._path, subset_file)
    names = set(open(p).read().splitlines()) if os.path.exists(p) else None
    return names

val_names, test_names = pick("validation_list.txt", 0), pick("testing_list.txt", 0)
paths = base._walker
def label_of(path): return os.path.basename(os.path.dirname(path))
def rel(path): return os.path.relpath(path, base._path)

train_p, test_p = [], []
for p in paths:
    if label_of(p) not in l2i: continue
    r = rel(p)
    if r in test_names: test_p.append(p)
    elif r not in val_names: train_p.append(p)
rng = np.random.default_rng(42); rng.shuffle(train_p); rng.shuffle(test_p)
train_p, test_p = train_p[:4000], test_p[:800]
print(len(train_p), len(test_p))

In [ ]:
SR = 16000
mel = torchaudio.transforms.MelSpectrogram(SR, n_fft=512, hop_length=160, n_mels=64)
to_db = torchaudio.transforms.AmplitudeToDB()

class SC(Dataset):
    def __init__(self, ps): self.ps = ps
    def __len__(self): return len(self.ps)
    def __getitem__(self, i):
        w, sr = torchaudio.load(self.ps[i])
        if sr != SR: w = torchaudio.functional.resample(w, sr, SR)
        w = nn.functional.pad(w, (0, SR - w.shape[1]))[:, :SR]       # pad / trim to 1 s
        return to_db(mel(w)), l2i[label_of(self.ps[i])]

train_dl = DataLoader(SC(train_p), batch_size=64, shuffle=True, num_workers=2)
test_dl = DataLoader(SC(test_p), batch_size=64, num_workers=2)
x, y = next(iter(train_dl)); print("batch:", x.shape)

plt.figure(figsize=(10, 3)); plt.imshow(x[0, 0].numpy(), origin="lower", aspect="auto"); plt.title(f"Mel spectrogram: '{LABELS[y[0]]}'"); plt.colorbar(); plt.show()

## 2. CNN classifier on mel-spectrograms

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, n=10):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(1, 16), blk(16, 32), blk(32, 64), nn.AdaptiveAvgPool2d(1))
        self.h = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, n))
    def forward(self, x): return self.h(self.f(x))

model = AudioCNN().to(device)
opt = torch.optim.Adam(model.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
tr_loss, te_acc = [], []
for ep in range(8):
    model.train(); tot = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device); opt.zero_grad(); l = crit(model(x), y); l.backward(); opt.step(); tot += l.item() * len(x)
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for x, y in test_dl: yp += model(x.to(device)).argmax(1).cpu().tolist(); yt += y.tolist()
    tr_loss.append(tot / len(train_dl.dataset)); te_acc.append(accuracy_score(yt, yp))
    print(f"epoch {ep+1}: loss {tr_loss[-1]:.4f}  test acc {te_acc[-1]:.4f}")

In [ ]:
print("Accuracy:", accuracy_score(yt, yp), "| Macro F1:", f1_score(yt, yp, average="macro"))
print(classification_report(yt, yp, target_names=LABELS))
plt.figure(figsize=(7, 6)); sns.heatmap(confusion_matrix(yt, yp), annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

## 3. Speech-to-Text with Whisper

In [ ]:
from transformers import pipeline
import jiwer, soundfile as sf
stt = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en", device=0 if device == "cuda" else -1)

def transcribe(path):
    w, sr = torchaudio.load(path)
    w = torchaudio.functional.resample(w.mean(0, keepdim=True), sr, SR)
    return stt({"raw": w.squeeze().numpy(), "sampling_rate": SR})["text"].strip().lower().strip(" .,!?")

# WER on test keywords (reference = the keyword itself)
sample = test_p[:100]
refs = [label_of(p) for p in sample]
hyps = [transcribe(p) for p in sample]
print("STT WER on Speech Commands:", round(jiwer.wer(refs, hyps), 4))
print(list(zip(refs[:8], hyps[:8])))

## 4. Text-to-Speech with gTTS, and a TTS -> STT round trip

In [ ]:
from gtts import gTTS
def speak(text, out="tts.mp3"):
    gTTS(text=text, lang="en").save(out); return out

sentences = ["The quick brown fox jumps over the lazy dog.", "Deep learning is fun.", "Turn the lights on.", "Please stop the music now."]
rt = []
for s in sentences:
    f = speak(s, "tts.mp3")
    rt.append(stt(f)["text"].strip())
print("Round-trip WER:", round(jiwer.wer([s.lower().strip('.') for s in sentences], [r.lower().strip(' .') for r in rt]), 4))
for s, r in zip(sentences, rt): print(f"IN : {s}\nOUT: {r}\n")
display(Audio(speak(sentences[0]), rate=SR))

## 5. End-to-end voice assistant demo
Take an audio clip -> classify the command -> transcribe -> reply with synthesized speech.

In [ ]:
def assistant(path):
    w, _ = torchaudio.load(path)
    w = nn.functional.pad(torchaudio.functional.resample(w, torchaudio.info(path).sample_rate, SR), (0, SR))[:, :SR]
    model.eval()
    with torch.no_grad(): probs = model(to_db(mel(w)).unsqueeze(0).to(device)).softmax(1)[0]
    cmd, conf = LABELS[int(probs.argmax())], float(probs.max())
    heard = transcribe(path)
    reply = f"I heard {heard}. Command detected: {cmd}, with {conf*100:.0f} percent confidence."
    print(reply); display(Audio(path)); display(Audio(speak(reply, "reply.mp3")))

assistant(test_p[0])

## Conclusion
A small CNN on mel-spectrograms classifies spoken keywords with high accuracy; Whisper provides robust STT (measured by WER) and gTTS completes the loop. The TTS->STT round-trip WER is a simple check of TTS intelligibility.